# Asistente de reglas del SRD 5.2.1

Ejecuta el código del repositorio. Requiere el secreto `ANTHROPIC_API_KEY`.

## 1. Preparación

In [ ]:
import os
if os.path.exists('/content/dnd_assistant'):
    !cd /content/dnd_assistant && git pull -q
else:
    !git clone -q https://github.com/MarioGavin/dnd_assistant.git /content/dnd_assistant
%cd /content/dnd_assistant
!pip -q install pymupdf rank-bm25 snowballstemmer anthropic pytest

# La clave de los Secretos pasa al entorno para que la lean los scripts
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

## 2. Tests

In [ ]:
!python -m pytest -q

## 3. Consultas

In [ ]:
!python -m asistente.responder "Tengo 2 niveles de cansancio, ¿qué me pasa al atacar?"

In [ ]:
!python -m asistente.responder "Dame la CA y los PG del contemplador"

In [ ]:
!python -m asistente.responder "Hemos pillado por sorpresa a unos bandidos, ¿pierden su primer turno?"

In [ ]:
# Validador de fichas: sin modelo, sin coste
!python -m asistente.fichas

## 4. Evaluación de la recuperación: BM25

In [ ]:
!python -m asistente.evaluar --solo-recuperacion --sin-vectores
!python -m asistente.evaluar --solo-recuperacion --sin-vectores --casos casos/consultas_contraste.json

## 5. Evaluación completa (consultas y fichas)

Tres ejecuciones por modelo: la API no admite `temperature`, así que la variabilidad se mide repitiendo.

In [ ]:
!python -m asistente.evaluar --modelo claude-haiku-5-5 --repeticiones 3

In [ ]:
!python -m asistente.evaluar --modelo claude-sonnet-5-5 --repeticiones 3

In [ ]:
# Configuración entregada: Haiku con la pregunta reformulada antes de buscar (DECISIONES.md, fila 14)
!python -m asistente.evaluar --modelo claude-haiku-5-5 --reformular --repeticiones 3

In [ ]:
# Opciones medidas (desactivadas por defecto), por orden de prioridad:
# contexto ampliado (60 candidatos, 20 fragmentos, 2 de 2014 y tablas del padre) y tablas del apartado padre (fallo 24)
!python -m asistente.evaluar --modelo claude-haiku-5-5 --reformular --contexto-amplio --repeticiones 3
!python -m asistente.evaluar --modelo claude-haiku-5-5 --tablas-padre --repeticiones 3
!python -m asistente.evaluar --modelo claude-haiku-5-5 --reformular --tablas-padre --repeticiones 3

In [ ]:
!python -m asistente.evaluar --tabla

## 6. Evaluación de la recuperación: combinada (opcional, GPU)

In [ ]:
!pip -q install sentence-transformers
!python -m asistente.evaluar --solo-recuperacion

In [ ]:
!python -m asistente.evaluar --solo-recuperacion --modelo-emb BAAI/bge-m3

## 7. Resultados

In [ ]:
!zip -qr resultados.zip resultados
from google.colab import files
files.download('resultados.zip')